# Corpus descriptives

Figure 1.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

import ast
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cfg.set_style()

books = pd.read_csv(cfg.BOOK_COUNTS).merge(pd.read_csv(cfg.BOOK_TOKENS),
                                           on="filename", how="left")
allow = set(cfg.ENGLISH_ORIGINALS.read_text().split()) - {""}

# One metadata row per file: a handful of titles match more than one row.
books = books[books["filename"].isin(allow)].drop_duplicates(subset="filename")
print(f"{len(books):,} works, {books['year'].min()}-{books['year'].max()}")
print(f"{books['total_sentences'].sum():,} sentences, "
      f"{books['n_tokens'].sum():,} tokens")
print(f"median book: {books['total_sentences'].median():,.0f} sentences")

## Authors

In [ ]:
def clean_author(value):
    """'Last, First' entries -> 'First Last'; joint authorship kept as one id."""
    value = str(value).strip()
    if value in ("[]", "", "nan"):
        return None
    names = re.findall(r"'([^']*)'", value)
    if not names:
        return None
    out = []
    for name in names:
        if "," in name:
            last, first = name.split(",", 1)
            out.append(f"{first.strip()} {last.strip()}")
        else:
            out.append(name.strip())
    return " & ".join(out)


books["author_clean"] = books["author"].apply(clean_author)
authored = books[books["author_clean"].notna()]
known_gender = books["author_gender"].isin(["male", "female"]).sum()

print(f"credited authors : {authored['author_clean'].nunique():,} "
      f"({len(books) - len(authored)} works uncredited)")
print(f"female-authored  : {(books['author_gender'] == 'female').sum():,} "
      f"({(books['author_gender'] == 'female').sum() / known_gender:.0%} of known)")

## Figure 1

In [ ]:
# Okabe-Ito colours: separable under the common forms of colour blindness.
MEASURES = [("books", "books", 1, "#0072B2"),
            ("sentences (10k)", "sentences", 1e4, "#D55E00"),
            ("tokens (100k)", "tokens", 1e5, "#009E73")]
DEC_LO, DEC_HI = 1780, 1920

shown = books[(books["decade"] >= DEC_LO) & (books["decade"] <= DEC_HI)]
volume = shown.groupby("decade").agg(books=("filename", "size"),
                                     sentences=("total_sentences", "sum"),
                                     tokens=("n_tokens", "sum"))

x = np.arange(len(volume))
width = 0.27
fig, ax = plt.subplots(figsize=(11.5, 5.2))
for k, (name, col, scale, colour) in enumerate(MEASURES):
    ax.bar(x + (k - 1) * width, volume[col] / scale, width=width, color=colour,
           label=name, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(volume.index, rotation=90, fontsize=9.5)
ax.set_xlabel("Decade")
ax.set_ylabel("Count (scaled)")
ax.legend(frameon=False, fontsize=10, loc="upper left")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.yaxis.grid(True, color=cfg.GRID, lw=0.9, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
cfg.save_fig(fig, "fig1_corpus_volume_by_decade")

print(f"shown: {len(shown):,} of {len(books):,} works "
      f"({DEC_LO}s-{DEC_HI}s)")
volume

## Predicted label distribution

In [ ]:
total = books["total_sentences"].sum()
totals = pd.Series({l: books[f"n_{l}"].sum() for l in cfg.LABELS})
pd.DataFrame({"sentences": totals,
              "% of corpus": (totals / total * 100).round(2)}
             ).sort_values("sentences", ascending=False)